# P3: MFCC

**Contenido del notebook.**
1. Implementación del pipeline MFCC sobre habla sintética.
2. Verificación computacional del rol de la transformada discreta del coseno (DCT).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig
from scipy.fft import dct
from IPython.display import Audio, display

plt.rcParams['figure.figsize'] = (10, 4)
np.set_printoptions(precision=3, suppress=True)

## Ejercicio 3. MFCC de una señal de habla sintética

In [ ]:
fs = 16000
F0 = 120
formantes = {
    ### COMPLETAR DICCIONARIO
}
anchos_BW = [80, 90, 120]

def sintetizar_vocal(formant_freqs, fs=16000, F0=120, duracion=0.35, bws=anchos_BW):
    N_x = int(duracion * fs)
    fuente = np.zeros(N_x)
    periodo = int(fs / F0)
    fuente[::periodo] = 1.0

    salida = fuente.copy()
    for f, bw in zip(formant_freqs, bws):
        r = np.exp(-np.pi * bw / fs)
        theta = 2 * np.pi * f / fs
        b = [1 - r*r]
        a = [1, -2*r*np.cos(theta), r*r]
        salida = sig.lfilter(b, a, salida)

    salida = salida / np.max(np.abs(salida)) 
    return salida

# Concatenar las cinco vocales para tener una señal con cambios espectrales.
audio = np.concatenate([
    ### COMPLETAR
])
print(f'Duración: {len(audio)/fs:.2f} s')
display(Audio(audio, rate=fs))

In [ ]:
def hz_to_mel(f):
    # COMPLETAR fórmula mel
    return None

def mel_to_hz(m):
    # COMPLETAR fórmula inversa
    return None

def enmarcar(senial, fs, win_ms=25, hop_ms=10):
    L = int(round(win_ms * 1e-3 * fs))
    H = int(round(hop_ms * 1e-3 * fs))
    n_frames = 1 + (len(senial) - L) // H
    frames = np.stack([senial[i*H:i*H+L] for i in range(n_frames)])
    ventana = np.hamming(L)
    return frames * ventana, L, H

def banco_mel(n_mels, n_fft, fs, f_min=0, f_max=None):
    if f_max is None:
        f_max = fs / 2

    # M filtros necesitan M+2 puntos de corte.
    puntos_mel = ### <-- COMPLETAR
    puntos_hz = ### <-- COMPLETAR
    bins = np.floor(n_fft * puntos_hz / fs).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)

    B = np.zeros((n_mels, n_fft//2 + 1))
    for m in range(1, n_mels + 1):
        izq, centro, der = bins[m-1], bins[m], bins[m+1]
        for k in range(izq, centro):
            B[m-1, k] = (k - izq) / max(centro - izq, 1)
        for k in range(centro, der):
            B[m-1, k] = (der - k) / max(der - centro, 1)
    return B, puntos_hz

def mfcc_propio(senial, fs, n_mfcc=13, n_mels=26, win_ms=25, hop_ms=10, n_fft=512):
    # 1. Preénfasis
    senial_pe = ### <-- COMPLETAR con alpha = 0.97

    # 2. Framing + ventana
    frames, L, H = enmarcar(senial_pe, fs, win_ms, hop_ms)

    # 3. Espectro de potencia
    X = np.fft.rfft(frames, n=n_fft, axis=1)
    S = ### <-- COMPLETAR

    # 4. Banco mel
    B, puntos_hz = banco_mel(n_mels, n_fft, fs, f_min=80, f_max=fs/2)
    E = ### <-- COMPLETAR

    # 5. Logaritmo
    logE = ### <-- COMPLETAR

    # 6. DCT
    C_ = ### <-- COMPLETAR
    C = ### <-- COMPLETAR

    return C, logE, S, B, puntos_hz

mfcc, logmel, S, B, puntos_hz = mfcc_propio(audio, fs)
print('log-mel:', logmel.shape)
print('MFCC:', mfcc.shape)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4),
    constrained_layout=True
)

im0 = axes[0].imshow(
    logmel.T,
    origin='lower',
    aspect='auto',
    interpolation='nearest',
    cmap='viridis'
)

axes[0].set_title('Energías log-mel')
axes[0].set_xlabel('Frame $t$')
axes[0].set_ylabel('Filtro mel $m$')

cbar0 = fig.colorbar(im0, ax=axes[0])
cbar0.set_label(r'$\log(E_m[t]+\varepsilon)$')


# MFCC con escala simétrica alrededor de cero
lim = np.max(np.abs(mfcc))

im1 = axes[1].imshow(
    mfcc.T,
    origin='lower',
    aspect='auto',
    interpolation='nearest',
    cmap='RdBu_r',
    vmin=-lim,
    vmax=lim
)

axes[1].set_title('MFCC')
axes[1].set_xlabel('Frame $t$')
axes[1].set_ylabel('Coeficiente $d$')

cbar1 = fig.colorbar(im1, ax=axes[1])
cbar1.set_label(r'$c_d[t]$')

plt.show()

**Interpretación:** <mark>COMPLETAR</mark>.

## Ejercicio 4. La DCT reduce correlaciones

Para justificar el uso de covarianzas diagonales en GMM, miramos las correlaciones entre dimensiones antes y después de la DCT.

In [ ]:
def media_abs_fuera_diagonal(C):
    mascara = ~np.eye(C.shape[0], dtype=bool)
    return np.mean(np.abs(C[mascara]))

corr_logmel = ### <-- COMPLETAR
corr_mfcc = ### <-- COMPLETAR

print('Correlación media fuera de diagonal, log-mel:', media_abs_fuera_diagonal(corr_logmel))
print('Correlación media fuera de diagonal, MFCC:', media_abs_fuera_diagonal(corr_mfcc))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(corr_logmel, vmin=-1, vmax=1, origin='lower')
axes[0].set_title('Correlación log-mel')
axes[0].set_xlabel('Filtro mel')
axes[0].set_ylabel('Filtro mel')

im1 = axes[1].imshow(corr_mfcc, vmin=-1, vmax=1, origin='lower')
axes[1].set_title('Correlación MFCC')
axes[1].set_xlabel('Coeficiente')
axes[1].set_ylabel('Coeficiente')

plt.colorbar(im1, ax=axes.ravel().tolist(), shrink=0.8)
plt.show()

In [ ]:
# Concentración de energía: varianza explicada por coeficiente MFCC
var_por_coef = ### <-- COMPLETAR
var_rel = var_por_coef / np.sum(var_por_coef)

plt.figure(figsize=(8, 3))
plt.stem(np.arange(len(var_rel)), var_rel)
plt.xlabel('Coeficiente MFCC')
plt.ylabel('Fracción de varianza')
plt.title('Concentración de variabilidad en los primeros coeficientes')
plt.grid(alpha=0.3)
plt.show()

**Conclusión:** <mark>COMPLETAR</mark>.